In [1]:
from pathlib import Path
import json
import pandas as pd

# 🔧 여기에 비교할 실험들 추가
EXPERIMENTS = {
    "repair": Path("../../results/phase2_qwen/humaneval/repair"),
    "plan": Path("../../results/phase2_qwen/humaneval/code_then_plan"),
    "plan_repair": Path("../../results/phase2_qwen/humaneval/code_then_plan_repair"),
    "policy": Path("../../results/phase2_qwen/humaneval/rpe_policy"),
    "policy_v2": Path("../../results/phase2_qwen/humaneval/rpe_policy_v2"),
}

def read_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


def compute_action_stats(step_df, traj_df, stage):
    total = len(traj_df)

    stage_df = step_df[step_df["stage"] == stage]

    used_tids = set(stage_df["trajectory_id"])
    used = len(used_tids)

    success_tids = set(stage_df[stage_df["status"] == "PASS"]["trajectory_id"])
    recovered = len(success_tids)

    calls = len(stage_df)
    call_success = len(stage_df[stage_df["status"] == "PASS"])

    return {
        "used": used,
        "recovered": recovered,
        "calls": calls,
        "call_success": call_success,
        "total": total,
    }


def print_action_stats(stats, stage):
    used = stats["used"]
    recovered = stats["recovered"]
    calls = stats["calls"]
    call_success = stats["call_success"]
    total = stats["total"]

    print("=" * 60)
    print(f"  [{stage}]")
    print(f"  [problem-level] 사용: {used}/{total} ({(used/total*100 if total else 0):.1f}%)")
    print(f"  [problem-level] 복구 성공: {recovered}/{used} ({(recovered/used*100 if used else 0):.1f}%)")
    print(f"  [call-level] 호출: {calls}, 성공: {call_success}/{calls} ({(call_success/calls*100 if calls else 0):.1f}%)")
    print("=" * 60)


# 🔥 전체 실험 루프
for name, result_dir in EXPERIMENTS.items():
    print("\n" + "#" * 70)
    print(f"### EXPERIMENT: {name}")
    print("#" * 70)

    step_path = result_dir / "step_logs.jsonl"
    traj_path = result_dir / "trajectory_logs.jsonl"

    if not step_path.exists():
        print("❌ step_logs 없음")
        continue

    step_df = read_jsonl(step_path)
    traj_df = read_jsonl(traj_path)

    # 가능한 stage 자동 탐색
    stages = sorted(step_df["stage"].dropna().unique())

    for stage in stages:
        stats = compute_action_stats(step_df, traj_df, stage)
        print_action_stats(stats, stage)


######################################################################
### EXPERIMENT: repair
######################################################################
  [generate]
  [problem-level] 사용: 164/164 (100.0%)
  [problem-level] 복구 성공: 107/164 (65.2%)
  [call-level] 호출: 164, 성공: 107/164 (65.2%)
  [repair]
  [problem-level] 사용: 57/164 (34.8%)
  [problem-level] 복구 성공: 28/57 (49.1%)
  [call-level] 호출: 628, 성공: 28/628 (4.5%)

######################################################################
### EXPERIMENT: plan
######################################################################
  [generate]
  [problem-level] 사용: 164/164 (100.0%)
  [problem-level] 복구 성공: 114/164 (69.5%)
  [call-level] 호출: 164, 성공: 114/164 (69.5%)
  [plan]
  [problem-level] 사용: 50/164 (30.5%)
  [problem-level] 복구 성공: 0/50 (0.0%)
  [call-level] 호출: 171, 성공: 0/171 (0.0%)
  [plan_code]
  [problem-level] 사용: 50/164 (30.5%)
  [problem-level] 복구 성공: 38/50 (76.0%)
  [call-level] 호출: 171, 성공: 38/171 (22.2%)

#########